In [70]:
! pip install cassandra-driver

In [71]:
import numpy as np

In [ ]:
from ssl import PROTOCOL_TLSv1_2, CERT_REQUIRED
import pandas as pd
import os
from cassandra.cluster import Cluster, ExecutionProfile, EXEC_PROFILE_DEFAULT, ProtocolVersion
from cassandra.auth import PlainTextAuthProvider
import json

ASTRA_DB_APPLICATION_TOKEN = "YOUR_TOKEN"

cloud_config = {
    'secure_connect_bundle': '/content/secure-connect-big-data-fruit.zip'
}

auth_provider = PlainTextAuthProvider("token", ASTRA_DB_APPLICATION_TOKEN)
profile = ExecutionProfile(request_timeout=30)

cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider, execution_profiles={EXEC_PROFILE_DEFAULT: profile})
session = cluster.connect('medallion')


In [73]:
df = pd.read_csv('sales_100.csv')

In [74]:
print("Columns in CSV:", df.columns.tolist())
print("\nFirst row data:")
print(df.iloc[0].to_dict())

Columns in CSV: ['Region', 'Country', 'Item Type', 'Sales Channel', 'Order Priority', 'Order Date', 'Order ID', 'Ship Date', 'UnitsSold', 'UnitPrice', 'UnitCost', 'TotalRevenue', 'TotalCost', 'TotalProfit']

First row data:
{'Region': 'Sub-Saharan Africa', 'Country': 'South Africa', 'Item Type': 'Fruits', 'Sales Channel': 'Offline', 'Order Priority': 'M', 'Order Date': '7/27/2012', 'Order ID': 443368995, 'Ship Date': '7/28/2012', 'UnitsSold': 1593, 'UnitPrice': 9.33, 'UnitCost': 6.92, 'TotalRevenue': 14862.69, 'TotalCost': 11023.56, 'TotalProfit': 3839.13}


In [75]:
rows = session.execute("""
    SELECT column_name, type FROM system_schema.columns
    WHERE keyspace_name = 'medallion' AND table_name = 'bronze_sales'
""")
for row in rows:
    print(f"{row.column_name}: {row.type}")

country: text
item_type: text
order_date: text
order_id: bigint
order_priority: text
region: text
sales_channel: text
ship_date: text
total_cost: decimal
total_profit: decimal
total_revenue: decimal
unit_cost: decimal
unit_price: decimal
units_sold: int


In [76]:
session.execute("TRUNCATE medallion.bronze_sales")

In [77]:
from cassandra.query import SimpleStatement

insert_query = """
INSERT INTO bronze_sales (
    order_id, region, country, item_type, sales_channel,
    order_priority, order_date, ship_date, units_sold,
    unit_price, unit_cost, total_revenue, total_cost, total_profit
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

prepared = session.prepare(insert_query)

for _, row in df.iterrows():
    session.execute(prepared, (
        int(row["Order ID"]),
        row["Region"],
        row["Country"],
        row["Item Type"],
        row["Sales Channel"],
        row["Order Priority"],
        row["Order Date"],
        row["Ship Date"],
        int(row["UnitsSold"]),
        float(row["UnitPrice"]),
        float(row["UnitCost"]),
        float(row["TotalRevenue"]),
        float(row["TotalCost"]),
        float(row["TotalProfit"])
    ))


In [78]:
from datetime import datetime

In [79]:
print("Column name details:")
for col in df.columns:
    print(f"'{col}' (type: {type(col)}, length: {len(col)})")

Column name details:
'Region' (type: <class 'str'>, length: 6)
'Country' (type: <class 'str'>, length: 7)
'Item Type' (type: <class 'str'>, length: 9)
'Sales Channel' (type: <class 'str'>, length: 13)
'Order Priority' (type: <class 'str'>, length: 14)
'Order Date' (type: <class 'str'>, length: 10)
'Order ID' (type: <class 'str'>, length: 8)
'Ship Date' (type: <class 'str'>, length: 9)
'UnitsSold' (type: <class 'str'>, length: 9)
'UnitPrice' (type: <class 'str'>, length: 9)
'UnitCost' (type: <class 'str'>, length: 8)
'TotalRevenue' (type: <class 'str'>, length: 12)
'TotalCost' (type: <class 'str'>, length: 9)
'TotalProfit' (type: <class 'str'>, length: 11)


In [80]:
df.columns = df.columns.str.strip()

In [81]:
df.head()

,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,UnitsSold,UnitPrice,UnitCost,TotalRevenue,TotalCost,TotalProfit
0,Sub-Saharan Africa,South Africa,Fruits,Offline,M,7/27/2012,443368995,7/28/2012,1593,9.33,6.92,14862.69,11023.56,3839.13
1,Middle East and North Africa,Morocco,Clothes,Online,M,9/14/2013,667593514,10/19/2013,4611,109.28,35.84,503890.08,165258.24,338631.84
2,Australia and Oceania,Papua New Guinea,Meat,Offline,M,5/15/2015,940995585,6/4/2015,360,421.89,364.69,151880.40,131288.40,20592.00
3,Sub-Saharan Africa,Djibouti,Clothes,Offline,H,5/17/2017,880811536,7/2/2017,562,109.28,35.84,61415.36,20142.08,41273.28
4,Europe,Slovakia,Beverages,Offline,L,10/26/2016,174590194,12/4/2016,3973,47.45,31.79,188518.85,126301.67,62217.18


In [84]:
session.execute("TRUNCATE medallion.silver_sales")

In [83]:
# Silver
df = df.rename(columns={
    "UnitsSold": "units_sold",
    "UnitPrice": "unit_price",
    "UnitCost": "unit_cost",
    "TotalRevenue": "total_revenue",
    "TotalCost": "total_cost",
    "TotalProfit": "total_profit",
    "Order ID": "order_id",
    "Order Date": "order_date",
    "Ship Date": "ship_date",
    "Item Type": "item_type",
    "Sales Channel": "sales_channel",
    "Order Priority": "order_priority",
    "Region": "region",
    "Country": "country"
})

# Convert date strings to datetime.date
df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%Y").dt.date
df["ship_date"] = pd.to_datetime(df["ship_date"], format="%m/%d/%Y").dt.date

# Calculate new fields
# profit_ratio = total_profit / total_revenue, delivery_days = (ship_date - order_date).days
df["profit_ratio"] = df["total_profit"] / df["total_revenue"]
df["delivery_days"] = (pd.to_datetime(df["ship_date"]) - pd.to_datetime(df["order_date"])).dt.days

df.head()

,region,country,item_type,sales_channel,order_priority,order_date,order_id,ship_date,units_sold,unit_price,unit_cost,total_revenue,total_cost,total_profit,profit_ratio,delivery_days
0,Sub-Saharan Africa,South Africa,Fruits,Offline,M,2012-07-27,443368995,2012-07-28,1593,9.33,6.92,14862.69,11023.56,3839.13,0.258307,1
1,Middle East and North Africa,Morocco,Clothes,Online,M,2013-09-14,667593514,2013-10-19,4611,109.28,35.84,503890.08,165258.24,338631.84,0.672035,35
2,Australia and Oceania,Papua New Guinea,Meat,Offline,M,2015-05-15,940995585,2015-06-04,360,421.89,364.69,151880.40,131288.40,20592.00,0.135580,20
3,Sub-Saharan Africa,Djibouti,Clothes,Offline,H,2017-05-17,880811536,2017-07-02,562,109.28,35.84,61415.36,20142.08,41273.28,0.672035,46
4,Europe,Slovakia,Beverages,Offline,L,2016-10-26,174590194,2016-12-04,3973,47.45,31.79,188518.85,126301.67,62217.18,0.330032,39


In [85]:
insert_query = """
INSERT INTO silver_sales (
    order_id, region, country, item_type, sales_channel, order_priority,
    order_date, ship_date, units_sold, unit_price, unit_cost, profit_ratio, delivery_days
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

prepared = session.prepare(insert_query)

for _, row in df.iterrows():
    session.execute(prepared, (
        int(row["order_id"]),
        row["region"],
        row["country"],
        row["item_type"],
        row["sales_channel"],
        row["order_priority"],
        row["order_date"],
        row["ship_date"],
        int(row["units_sold"]),
        float(row["unit_price"]),
        float(row["unit_cost"]),
        float(row["profit_ratio"]),
        int(row["delivery_days"])
    ))

In [86]:
# Gold 1: Total profit & average profit ratio per region & item_type

gold_regional_profits = df.groupby(["region", "item_type"]).agg({
    "total_profit": "sum",
    "profit_ratio": "mean",
    "units_sold": "sum"
}).reset_index()

gold_regional_profits.columns = ["region", "item_type", "total_profit", "avg_profit_ratio", "total_units_sold"]
gold_regional_profits.head()


,region,item_type,total_profit,avg_profit_ratio,total_units_sold
0,Asia,Beverages,143351.64,0.330032,9154
1,Asia,Cereal,1206507.21,0.430676,13619
2,Asia,Fruits,26432.88,0.258307,10968
3,Asia,Household,1494221.68,0.247999,9016
4,Asia,Meat,1501156.80,0.135580,26244


In [87]:
session.execute("TRUNCATE medallion.gold_regional_profits")

insert_query = """
INSERT INTO gold_regional_profits (
    region, item_type, total_profit, avg_profit_ratio, total_units_sold
) VALUES (?, ?, ?, ?, ?)
"""

prepared = session.prepare(insert_query)

for _, row in gold_regional_profits.iterrows():
    session.execute(prepared, (
        row["region"],
        row["item_type"],
        float(row["total_profit"]),
        float(row["avg_profit_ratio"]),
        int(row["total_units_sold"])
    ))

In [88]:
# Gold 2: Averge delivery days, total revenue, order count per sales channel (Online or offline)

gold_channel = df.groupby("sales_channel").agg({
    "delivery_days": "mean",
    "total_revenue": "sum",
    "order_id": "count"
}).reset_index()

gold_channel.columns = ["sales_channel", "avg_delivery_days", "total_revenue", "order_count"]
gold_channel.head()


,sales_channel,avg_delivery_days,total_revenue,order_count
0,Offline,23.475000,60061793.39,40
1,Online,25.305085,84628184.37,59


In [89]:
insert_query = """
INSERT INTO gold_channel_performance (
    sales_channel, avg_delivery_days, total_revenue, order_count
) VALUES (?, ?, ?, ?)
"""

prepared = session.prepare(insert_query)

for _, row in gold_channel.iterrows():
    session.execute(prepared, (
        row["sales_channel"],
        float(row["avg_delivery_days"]),
        float(row["total_revenue"]),
        int(row["order_count"])
    ))


In [90]:
# Gold 3: Average profit per unit, total volume, region count per item_type
gold_top_products = df.groupby("item_type").agg({
    "total_profit": "sum",
    "units_sold": "sum",
    "region": pd.Series.nunique
}).reset_index()

gold_top_products["avg_profit_per_unit"] = gold_top_products["total_profit"] / gold_top_products["units_sold"]
gold_top_products.columns = ["item_type", "total_profit", "total_volume", "region_count", "avg_profit_per_unit"]
gold_top_products.head()


,item_type,total_profit,total_volume,region_count,avg_profit_per_unit
0,Baby Food,1952859.92,20372,4,95.86
1,Beverages,707925.96,45206,6,15.66
2,Cereal,4055295.84,45776,3,88.59
3,Clothes,2948469.12,40148,4,73.44
4,Cosmetics,11424476.09,65707,4,173.87


In [91]:
insert_query = """
INSERT INTO gold_top_products (
    item_type, avg_profit_per_unit, total_volume, region_count
) VALUES (?, ?, ?, ?)
"""

prepared = session.prepare(insert_query)

for _, row in gold_top_products.iterrows():
    session.execute(prepared, (
        row["item_type"],
        float(row["avg_profit_per_unit"]),
        int(row["total_volume"]),
        int(row["region_count"])
    ))
